# (1) load to dataFrame

In [283]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import seaborn as sns
import matplotlib.pyplot as plt #导入


# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

import os
print(os.listdir("../input"))

df_train = pd.read_csv('../input/train.csv')
df_test = pd.read_csv('../input/test.csv')
df_sub=pd.read_csv('../input/gender_submission.csv')

# Any results you write to the current directory are saved as output.

['train.csv', 'gender_submission.csv', 'test.csv']
<class 'pandas.core.frame.DataFrame'>


# (2) explore the data

In [284]:
df_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# (3) clean and digitize the data

In [285]:
df_train=df_train.drop(['PassengerId','Ticket','Name','Embarked'],axis=1) 
df_test=df_test.drop(['PassengerId','Ticket','Name','Embarked'],axis=1) 


In [287]:
df_test.mean()

Pclass     2.265550
Age       30.272590
SibSp      0.447368
Parch      0.392344
Fare      35.627188
dtype: float64

In [289]:
df_test=df_test.fillna(df_test.mean())
#df_test
df_train=df_train.fillna(df_train.mean())
#df_train

In [290]:
m_train=df_train.values
m_test=df_test.values

m_train[m_train[:,2]=='male',2]=1
m_train[m_train[:,2]=='female',2]=0


m_test[m_test[:,1]=='male',1]=1
m_test[m_test[:,1]=='female',1]=0

In [291]:
cabins=list(set(m_train[:,7]))
cabinMap={}
for i in range(len(cabins)):
    cabinMap[cabins[i]]=i
    
for i in range(m_train.shape[0]):
    m_train[i,7]=cabinMap[m_train[i,7]]

for i in range(m_test.shape[0]):
    if m_test[i,6] in cabinMap:
        m_test[i,6]=cabinMap[m_test[i,6]]
    else:
        cabinMap[m_test[i,6]]=len(cabinMap)
        m_test[i,6]=cabinMap[m_test[i,6]]

In [307]:


x=m_train[:,1:]
y=m_train[:,0]

xt=m_test

y=y.astype('int')

# (4) machine learning

## (4.1) all females are survived

In [ ]:
test = pd.read_csv('../input/test.csv')
test['Survived'] = 0
test.loc[test['Sex'] == 'female','Survived'] = 1
data_to_submit = pd.DataFrame({
    'PassengerId':test['PassengerId'],
    'Survived':test['Survived']
})


## (4.2) logisticRegression

In [310]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(x,y)

yt=logreg.predict(xt)


/opt/conda/lib/python3.6/site-packages/sklearn/linear_model/logistic.py:433: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


In [311]:
test = pd.read_csv('../input/test.csv')
test['Survived'] = yt
data_to_submit = pd.DataFrame({
    'PassengerId':test['PassengerId'],
    'Survived':test['Survived']
})

In [312]:
data_to_submit

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


# (5) assessing models and submit

In [313]:
data_to_submit.to_csv('csv_to_submit.csv', index = False)